# Compare CTRL and DART hindcast ensemble spatial ranges

Opens configurable CTRL and DART hindcast ensembles, caches each experiment's member spatial minima, means, and maxima, and overlays their member traces, ensemble envelopes, and ensemble means in one comparison figure.


In [ ]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")

source_type = "member_netcdf"  # "member_netcdf", "netcdf", "zarr", or "intake_esm"
EXPERIMENTS = {
    "CTRL10-S0": {
        "label": "CTRL hindcast (ENS=10)",
        "source": str(
            INPUT_DATA_ROOT
            / "CTRLEN10_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy"
            / "EN*/archive/post/atm/180x360_aave/ts/daily"
            / "TREFHT.EN*.2012.nc"
        ),
        "cache_tag": "CTRL10_S0_TREFHT_201201_201202",
        "color": "tab:blue",
    },
    "DART20-S0": {
        "label": "DART hindcast (ENS=20)",
        "source": str(
            INPUT_DATA_ROOT
            / "DARTEN20_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy"
            / "EN*/archive/post/atm/180x360_aave/ts/daily"
            / "TREFHT.EN*.2012.nc"
        ),
        "cache_tag": "DART20_S0_TREFHT_201201_201202",
        "color": "tab:red",
    },
    # Optional alternative/additional comparison:
    # "DART40-S0": {
    #     "label": "DART hindcast (ENS=40)",
    #     "source": str(
    #         INPUT_DATA_ROOT
    #         / "DARTEN40_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy"
    #         / "EN*/archive/post/atm/180x360_aave/ts/daily"
    #         / "TREFHT.EN*.2012.nc"
    #     ),
    #     "cache_tag": "DART40_S0_TREFHT_201201_201202",
    #     "color": "tab:green",
    # },
}

storage_options = {}
dataset_key = None  # optional Intake-ESM dataset key
variable = "TREFHT"
member_dim = "member_id"
time_dim = "time"
time_slice = ("2012-01-01", "2012-02-29")  # None retains every input time
spatial_dims = None  # None infers every non-member, non-time dimension
level_dim = None  # inferred from lev, plev, or level when present
level = None  # None selects the last model level when the variable is 3-D
force_compute = False
show_members = True
show = True


In [ ]:
import sys

import matplotlib.pyplot as plt

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))

from configs.output_paths import workflow_output_dirs
from util.ensemble_summary import (
    open_ensemble_source,
    cache_ensemble_summary,
    plot_ensemble_summary_comparison,
)


## Notes: experiments, variables, and metrics

Edit `EXPERIMENTS` to select any two or more hindcasts. The default comparison is CTRL10-S0 versus DART20-S0; a ready-to-enable DART40-S0 entry is also provided. Each entry defines its label, source, request-keyed cache tag, and plot color.

For `member_netcdf`, each source is a glob over files containing `EN##` in the filename, and the notebook builds `member_dim` automatically. Local NetCDF and optional Zarr inputs support any variable containing the configured member and time dimensions plus at least one spatial dimension. The optional Intake-ESM mode accepts any catalog variable but requires `intake-esm`. Three-dimensional variables may select a positional level with an integer or a coordinate value with a float.

Cached member metrics are `spatial_min`, `spatial_mean`, and `spatial_max`. Their experiment-level envelope and center fields are `ensemble_min`, `ensemble_mean`, and `ensemble_max`, each indexed by spatial statistic. Existing per-experiment caches are reused without opening raw data unless `force_compute = True`.

The comparison figure uses a distinct color for each experiment. Faint lines show individual members when `show_members = True`, shaded regions show each ensemble's minimum-to-maximum envelope, and solid lines show ensemble means.


In [ ]:
data_dir, figure_dir = workflow_output_dirs(
    "analysis_atm_da", "ensemble_ranges", output_root=DIAGNOSTIC_OUTPUT_ROOT
)
if len(EXPERIMENTS) < 2:
    raise ValueError("Configure at least two experiments for a comparison")

summaries = {}
try:
    for experiment, config in EXPERIMENTS.items():
        cache_path = data_dir / f"{config['cache_tag']}_ensemble_summary.nc"
        existed = cache_path.is_file()
        dataset = None
        if force_compute or not existed:
            dataset = open_ensemble_source(
                config["source"],
                source_type=config.get("source_type", source_type),
                variable=variable,
                dataset_key=config.get("dataset_key", dataset_key),
                storage_options=config.get("storage_options", storage_options),
                member_dim=member_dim,
            )
            if time_slice is not None:
                if time_dim not in dataset.coords and time_dim not in dataset.dims:
                    raise ValueError(
                        f"{experiment} source has no {time_dim!r} coordinate"
                    )
                dataset = dataset.sel({time_dim: slice(*time_slice)})

        try:
            summary = cache_ensemble_summary(
                cache_path,
                dataset=dataset,
                force_compute=force_compute,
                variable=variable,
                member_dim=member_dim,
                time_dim=time_dim,
                spatial_dims=spatial_dims,
                level_dim=level_dim,
                level=level,
            )
            summaries[experiment] = summary.load()
        finally:
            if dataset is not None:
                dataset.close()

        action = "[REUSE]" if existed and not force_compute else "[WRITE]"
        print(action, cache_path)

    labels = {
        experiment: config.get("label", experiment)
        for experiment, config in EXPERIMENTS.items()
    }
    colors = {
        experiment: config["color"]
        for experiment, config in EXPERIMENTS.items()
        if "color" in config
    }
    comparison_name = " vs ".join(labels.values())
    figure = plot_ensemble_summary_comparison(
        summaries,
        labels=labels,
        colors=colors,
        title=f"{variable} ensemble spatial ranges: {comparison_name}",
        show_members=show_members,
    )
    comparison_tag = "_vs_".join(EXPERIMENTS)
    figure_path = (
        figure_dir
        / f"{comparison_tag}_{variable}_ensemble_summary_comparison.png"
    )
    figure.savefig(figure_path, dpi=200, bbox_inches="tight")
    if not show:
        plt.close(figure)
    print("[FIGURE]", figure_path)
finally:
    for summary in summaries.values():
        summary.close()
